# Silver → Gold: dim_province

**Propósito:** Crear la dimensión de provincias extrayéndola de `dbo.salestrack_sales_final` en Silver.
No existe como tabla propia en Bronze — se deriva de `province_id` + `province_name`.

**Nota:** `province_id` en fact_sales equivale a `region_id` en dim_customer (mismo código de provincia español).
Esta tabla centraliza la dimensión geográfica.

**Columnas resultantes:**
- `province_id` (int) → clave primaria (código provincia español, ej. 8 = Barcelona)
- `province_name` → nombre de la provincia

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [ ]:
%run ./config

In [ ]:
SILVER_TABLE = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
GOLD_TABLE   = f"{DEFAULT_SCHEMA}.dim_province"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas fuente (salestrack): {df_silver.count()}")

In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
# Si un province_id tiene varios nombres (datos sucios), tomamos el más frecuente
w = Window.partitionBy("province_id").orderBy(F.col("freq").desc())

df_gold = (
    df_silver
    .filter(
        F.col("province_id").isNotNull() &
        F.col("province_name").isNotNull()
    )
    .select("province_id", "province_name")
    .groupBy("province_id", "province_name")
    .agg(F.count("*").alias("freq"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .drop("freq", "rank")
    .withColumn("province_name", F.trim(F.initcap(F.col("province_name"))))
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
    .orderBy("province_id")
)

In [ ]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count = df_gold.count()
null_ids  = df_gold.filter(F.col("province_id").isNull()).count()
dup_ids   = df_gold.groupBy("province_id").count().filter(F.col("count") > 1).count()

print(f"Provincias únicas en Gold : {row_count}")
print(f"province_id nulos         : {null_ids}")
print(f"province_id duplicados    : {dup_ids}")

assert null_ids == 0, "ERROR: hay province_id nulos"
assert dup_ids  == 0, "ERROR: hay province_id duplicados tras deduplicación"

df_gold.show(20, truncate=False)

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"Tabla {GOLD_TABLE} escrita en {GOLD_LAKEHOUSE} con {row_count} filas.")